## Overview ##

####
Alzheimer's disease and Alzheimer's disease related dementias (AD/ADRD) are a set of brain disorders affecting more than 6 million Americans. Early intervention is crucial for successful disease modification, but detecting early signs of cognitive decline and AD/ADRD remains challenging. Current clinical methods often lack the sensitivity needed for early prediction, especially in underrepresented groups.

The objective of this challenge track is to improve early prediction of Alzheimer's disease and related dementias (AD/ADRD) using acoustic biomarkers from voice recordings. Through this initiative, the National Institute on Aging (NIA) aims to improve accuracy across diverse populations and explore understudied factors that may indicate early AD/ADRD.
####

## Problem description ##

####
The challenge is centered around developing better methods for prediction of Alzheimer's disease and Alzheimer's disease related dementias (AD/ADRD) as early as possible. Phase 2 — [Build IT!]: Algorithms and Approaches Acoustic Track — is focused on building innovative models for early detection of AD/ADRD using audio data.

Current methods of screening for AD/ADRD are time intensive and difficult to perform. Models that can flag individuals with a high likelihood of cognitive decline early based on vocal characteristics have the potential to catch and treat cognitive decline earlier, and to reduce disparities in care for marginalized groups. Speech data would be a highly cost-effective and noninvasive method for assessing cognitive decline.

Overview of the data files provided for this competitions:
####

## Imports ##

In [1]:
import tensorflow as tf

import pandas as pd
import numpy as np

from google.colab import files
import zipfile
from glob import glob

# Data manipulation
import librosa
import numpy as np
import pandas as pd
import os

# Machine Learning

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    log_loss,
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

# Data Visualization
import librosa.display
import matplotlib.pyplot as plt

from scipy.fftpack import dct

import glob

print(tf.__version__)

2.19.0


In [2]:
!python --version

Python 3.12.12


In [3]:
! pip install xgboost

## Mount Google drive ##

In [4]:
# Mount folders from google drive
from google.colab import drive

DRIVE_PATH = '/content/drive'
drive.mount(DRIVE_PATH)

Mounted at /content/drive


## Directories / file paths Constants ##

In [ ]:
# audio_files = list()

# MYDRIVE_PATH = os.path.join(DRIVE_PATH, 'MyDrive/')
# PROJECT_DIR = os.path.join(MYDRIVE_PATH, 'dsfs-ft-31', 'alzheimer_data/')
# print('Project directory : ', PROJECT_DIR)
# TRAIN_AUDIOS_SAMPLE_DIR = os.path.join(PROJECT_DIR + 'train_audios_sample/')
# TRAIN_AUDIOS_ZIP_FNAME ='train_audios.zip'
# TEST_AUDIOS_ZIP_FNAME ='test_audios.zip'
# TRAIN_AUDIOS_SAMPLE_ZIP_FNAME ='train_audios_sample.zip'

# TRAIN_AUDIOS_DIR = os.path.join(PROJECT_DIR , 'train_audios/')
# TEST_AUDIOS_DIR = os.path.join(PROJECT_DIR, 'test_audios', 'mp3/')

# TRAIN_AUDIOS_WAV_DIR = os.path.join(PROJECT_DIR , 'train_audios','wav/')
# TEST_AUDIOS_WAV_DIR = os.path.join(PROJECT_DIR, 'test_audios','wav/')

# MODEL_DIR = os.path.join(PROJECT_DIR, 'Model/')

# path_to_zip_file = TRAIN_AUDIOS_SAMPLE_ZIP_FNAME

# LABELS_DIR = os.path.join(PROJECT_DIR, 'labels/')
# TRAIN_LABELS_FILE_PATH = LABELS_DIR + 'train_labels.csv'
# print(TRAIN_LABELS_FILE_PATH)

Project directory :  /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/
/content/drive/MyDrive/dsfs-ft-31/alzheimer_data/labels/train_labels.csv


In [5]:
DRIVE_PATH = '/content/drive'
MYDRIVE_PATH = os.path.join(DRIVE_PATH, 'MyDrive/')
TARGET_DIR = os.path.join(MYDRIVE_PATH, 'dsfs-ft-31', 'alzheimer_data/')
TRAIN_AUDIOS_WAV_DIR = os.path.join(TARGET_DIR , 'train_audios','wav/')
print(f'TRAIN_AUDIOS_WAV_DIR : {TRAIN_AUDIOS_WAV_DIR}')
TEST_AUDIOS_WAV_DIR = os.path.join(TARGET_DIR, 'test_audios','wav/')
LABELS_DIR = os.path.join(TARGET_DIR, 'labels/')
TRAIN_LABELS_FILE_PATH = LABELS_DIR + 'train_labels.csv'
print(f'TRAIN_LABELS_FILE_PATH : {TRAIN_LABELS_FILE_PATH}')
TEST_LABELS_FILE_PATH = LABELS_DIR + 'test_labels.csv'
METADATA_FILE_PATH = os.path.join(TARGET_DIR, 'metadata', 'metadata.csv')
print(f'METADATA_FILE_PATH : {METADATA_FILE_PATH}')

TRAIN_FEATURES_FILE_PATH = os.path.join(TARGET_DIR, 'features', 'train_features.csv')
print(f'TRAIN_FEATURES_FILE_PATH : {TRAIN_FEATURES_FILE_PATH}')

TRAIN_AUDIOS_WAV_DIR : /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/train_audios/wav/
TRAIN_LABELS_FILE_PATH : /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/labels/train_labels.csv
METADATA_FILE_PATH : /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/metadata/metadata.csv
TRAIN_FEATURES_FILE_PATH : /content/drive/MyDrive/dsfs-ft-31/alzheimer_data/features/train_features.csv


## Data Processing Constants ##

In [6]:
# Constants
# Pre-processing constants - values recommended for human speech features extraction
SAMPLING_RATE = 16000
AUDIO_DURATION = 30
N_MFCC = 40
# Get classes (diagnosis_control, diagnosis_mci, diagnosis_adrd)
CLASSES = ['diagnosis_control', 'diagnosis_mci', 'diagnosis_adrd']

### Loading Labels / y dataSet ##

In [7]:
labels_df = pd.read_csv(TRAIN_LABELS_FILE_PATH)
labels_df.head(5)

,uid,diagnosis_control,diagnosis_mci,diagnosis_adrd
0,aaop,0.0,1.0,0.0
1,abgk,1.0,0.0,0.0
2,ablf,1.0,0.0,0.0
3,acad,1.0,0.0,0.0
4,acis,0.0,1.0,0.0


In [8]:
# Missing values
(len(labels_df.index) - labels_df.isna().count()).to_frame().T

,uid,diagnosis_control,diagnosis_mci,diagnosis_adrd
0,0,0,0,0


In [9]:
# Class distribution
#classes_count_dtf = train_labels_df.sum(numeric_only=True).to_frame().T.rename_axis('Total')
classes_count_dtf = labels_df.sum(numeric_only=True).to_frame()
classes_count_dtf.rename(columns={0:'Total'}, inplace=True)
classes_count_dtf.head(5)
classes_count_dtf['prop']=classes_count_dtf['Total']*100/classes_count_dtf['Total'].sum()
classes_count_dtf.head(5)

,Total,prop
diagnosis_control,911.0,55.346294
diagnosis_mci,217.0,13.183475
diagnosis_adrd,518.0,31.470231


### Processing y target column (class name) ###

In [10]:

#labels_df['target'] = labels_df[['diagnosis_control', 'diagnosis_mci', 'diagnosis_adrd']].values.tolist()
labels_df['target'] = labels_df[CLASSES].idxmax(axis=1)
labels_df.head(5)

,uid,diagnosis_control,diagnosis_mci,diagnosis_adrd,target
0,aaop,0.0,1.0,0.0,diagnosis_mci
1,abgk,1.0,0.0,0.0,diagnosis_control
2,ablf,1.0,0.0,0.0,diagnosis_control
3,acad,1.0,0.0,0.0,diagnosis_control
4,acis,0.0,1.0,0.0,diagnosis_mci


## Generating X features (audio signal Pre-Processing) ##

In [11]:
class Sound_processing_config:
    def __init__(self, sampling_rate=16000, audio_duration=30, n_mfcc=40, n_fft=512):
        self.sampling_rate = sampling_rate
        self.audio_duration = audio_duration
        self.n_mfcc = n_mfcc
        self.audio_length = self.sampling_rate * self.audio_duration
        self.dim = (self.n_mfcc, 1 + int(np.floor(self.audio_length/512)), 1)
        self.n_fft = n_fft

    def get_dim(self):
        return self.dim

    def get_audio_length(self):
        return self.audio_length

    def get_sampling_rate(self):
        return self.sampling_rate

    def get_audio_duration(self):
        return self.audio_duration

    def get_n_mfcc(self):
        return self.n_mfcc

    def get_n_fft(self):
        return self.n_fft

sound_processing_config = Sound_processing_config(sampling_rate=SAMPLING_RATE, audio_duration=AUDIO_DURATION, n_mfcc=N_MFCC)

#### Audio file features extraction #####


In [12]:
def extract_audio_features(file_path, sound_processing_config: Sound_processing_config):
  y, sr = librosa.load(file_path, sr=sound_processing_config.get_sampling_rate())
  nmfcc = sound_processing_config.get_n_mfcc()
  nfft = sound_processing_config.get_n_fft()
  features = {
        'chroma_stft_var': np.var(librosa.feature.chroma_stft(y=y, sr=sr)),
        'chroma_stft_mean': np.mean(librosa.feature.chroma_stft(y=y, sr=sr)),
        'chroma_cqt_var': np.var(librosa.feature.chroma_cqt(y=y, sr=sr)),
        'chroma_cqt_mean': np.mean(librosa.feature.chroma_cqt(y=y, sr=sr)), # CQT (Constant-Q Transform)
        'rmse_var': np.var(librosa.feature.rms(y=y)),
        'rmse_mean': np.mean(librosa.feature.rms(y=y)),
        'spectral_centroid_var': np.var(librosa.feature.spectral_centroid(y=y, sr=sr)),
        'spectral_centroid_mean': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
        'spectral_bandwidth_var': np.var(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
        'spectral_bandwidth_mean': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
        'rolloff_var': np.var(librosa.feature.spectral_rolloff(y=y, sr=sr)),
        'rolloff_mean': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),
        'zero_crossing_rate_var': np.var(librosa.feature.zero_crossing_rate(y)),
        'zero_crossing_rate_mean': np.mean(librosa.feature.zero_crossing_rate(y)),
        'mfcc1_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc1_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[0]),
        'mfcc2_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc2_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[1]),
        'mfcc3_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc3_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[2]),
        'mfcc4_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc4_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[3]),
        'mfcc5_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc5_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[4]),
        'mfcc6_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc6_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[5]),
        'mfcc7_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc7_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[6]),
        'mfcc8_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc8_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[7]),
        'mfcc9_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc9_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[8]),
        'mfcc10_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc10_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[9]),
        'mfcc11_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc11_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[10]),
        'mfcc12_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc12_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[11]),
        'mfcc13_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc13_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[12]),
        'mfcc14_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc14_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[13]),
        'mfcc15_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc15_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[14]),
        'mfcc16_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc16_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[15]),
        'mfcc17_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc17_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[16]),
        'mfcc18_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc18_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[17]),
        'mfcc19_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc19_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[18]),
        'mfcc20_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc20_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[19]),
        'mfcc21_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc21_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[20]),
        'mfcc22_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc22_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[21]),
        'mfcc23_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc23_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[22]),
        'mfcc24_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc24_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[23]),
        'mfcc25_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc25_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[24]),
        'mfcc26_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc26_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[25]),
        'mfcc27_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc27_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[26]),
        'mfcc28_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc28_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[27]),
        'mfcc29_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc29_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[28]),
        'mfcc30_var' : np.var(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=nmfcc,n_fft=nfft)),
        'mfcc30_mean': np.mean(librosa.feature.mfcc(y=y, sr=sr,n_mfcc=nmfcc,n_fft=nfft)[29]),
        'duration': librosa.get_duration(y=y, sr=sr)

        # Ajouter plus de caractéristiques si nécessaire
    }
  items = list(features.items())
  items.insert(0, ('uid',os.path.splitext(os.path.basename(file_path))[0]))
  return dict(items)

##### Testing audio features extraction (on a sample)#####

In [13]:
import glob
audio_files_paths = glob.glob(TRAIN_AUDIOS_WAV_DIR + '*.wav')

SAMPLE_AUDIO_FILE_PATH = audio_files_paths[0]
audio_features = extract_audio_features(SAMPLE_AUDIO_FILE_PATH, sound_processing_config)
print(type(audio_features))
print(list(audio_features.items())[:10])

<class 'dict'>
[('uid', 'bldi'), ('chroma_stft_var', np.float32(0.104857035)), ('chroma_stft_mean', np.float32(0.33570942)), ('chroma_cqt_var', np.float32(0.07974357)), ('chroma_cqt_mean', np.float32(0.41907954)), ('rmse_var', np.float32(1.5271331e-05)), ('rmse_mean', np.float32(0.009867816)), ('spectral_centroid_var', np.float64(296694.7959000072)), ('spectral_centroid_mean', np.float64(970.036167319162)), ('spectral_bandwidth_var', np.float64(132404.00118168365))]


#### Generate X - Pandas DataFrame with sound features ####

In [14]:
def process_X_features(labels_df: pd.DataFrame, audio_file_dir:str, sound_processing_config: Sound_processing_config) -> pd.DataFrame:
  """
  """
  dict_list = list()
  for idx, row in labels_df.iterrows():
    uid = row['uid']
    file_path = os.path.join(TRAIN_AUDIOS_WAV_DIR, f"{uid}.wav")
    dict_list.append(extract_audio_features(file_path, sound_processing_config))
  return pd.DataFrame(dict_list).drop(columns=['uid'], axis=1)


#### X generation ####

In [15]:
X = process_X_features(labels_df, TRAIN_AUDIOS_WAV_DIR, sound_processing_config)
X.head(5)
print(len(X))

/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=279
  warnings.warn(


1646


#### (Optional) persist X DataFrame if necessary (cause X generation takes time) ####

In [ ]:
# saving X DataFrame if necessary (cause X generation takes time)
X_DTF_PATH = os.path.join(PROJECT_DIR, 'X_features_RandomForest_idx_18122024.csv')
#X.to_csv(X_DTF_PATH, encoding='utf-8', index=True)

#### (Optional) Mount X DataFrame if necessary ####

In [ ]:
# mounting X DataFrame if necessary
X = pd.read_csv(X_DTF_PATH)
X.head(5)
X.columns
X = X.drop(columns=['Unnamed: 0'], axis=1)
X.head(5)

,chroma_stft_var,chroma_stft_mean,chroma_cqt_var,chroma_cqt_mean,rmse_var,rmse_mean,spectral_centroid_var,spectral_centroid_mean,spectral_bandwidth_var,spectral_bandwidth_mean,...,mfcc26_mean,mfcc27_var,mfcc27_mean,mfcc28_var,mfcc28_mean,mfcc29_var,mfcc29_mean,mfcc30_var,mfcc30_mean,duration
0,0.105255,0.270625,0.089752,0.373157,0.004778,0.091641,471129.910652,1260.074001,183767.171962,1308.565427,...,3.998443,5272.2640,-1.382246,5272.2640,2.144701,5272.2640,-1.292032,5272.2640,-3.201964,30.000000
1,0.104901,0.476995,0.057080,0.514384,0.000001,0.005514,52640.257432,861.857059,45202.154380,1616.547899,...,7.631622,11860.5050,2.175818,11860.5050,5.014473,11860.5050,2.108671,11860.5050,7.806511,14.992000
2,0.102833,0.393842,0.076787,0.447088,0.000003,0.001389,579277.918387,1462.336711,261758.188994,1749.623960,...,-0.053288,16449.4000,-4.094741,16449.4000,-1.203276,16449.4000,-4.103865,16449.4000,-1.100633,17.988000
3,0.097296,0.370170,0.074803,0.479214,0.001746,0.058786,495789.415308,1507.227050,104817.467459,1799.560181,...,0.339511,4817.6157,-0.052488,4817.6157,2.990898,4817.6157,-3.191279,4817.6157,1.849848,30.000062
4,0.098340,0.321130,0.081828,0.380042,0.001860,0.079338,631324.066954,1759.882280,120467.652215,1865.912475,...,-2.930300,3430.3770,-6.401167,3430.3770,-4.676897,3430.3770,-5.568949,3430.3770,-0.704274,30.000062


### Audio files Pre-processing Checks ###

In [21]:
print(X.shape)
print(y.shape)
print(type(y))
print(y[:2])

(1646, 75)
(1646,)
<class 'pandas.core.series.Series'>
0        diagnosis_mci
1    diagnosis_control
Name: target, dtype: object


# Train / Validation X, y preprocessing & split

In [16]:
# read the different data types in X to apply relevant transformer
y = labels_df['target']
X.dtypes.value_counts()

,count
float32,66
float64,9


In [17]:
y.value_counts()/y.shape[0]

,count
target,
diagnosis_control,0.553463
diagnosis_adrd,0.314702
diagnosis_mci,0.131835


In [32]:
#Stratify
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y )
print(type(X_val))

<class 'pandas.core.frame.DataFrame'>


In [33]:
y_train[:3]

,target
746,diagnosis_control
321,diagnosis_control
83,diagnosis_control


#### Label encode target (y) ####

In [34]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_val_encoded = encoder.transform(y_val)
print(y_train_encoded[:3])
print(y_val_encoded[:3])

[1 1 1]
[2 0 0]


In [23]:
numeric_features = []
categorical_features = []

for col_name in X.columns:
    dtype = X[col_name].dtype
    if dtype in ['int64', 'float64', 'float32']:
        numeric_features.append(col_name)
    elif dtype == 'bool':
        numeric_features.append(col_name)
    else:
        categorical_features.append(col_name)

print('Found numeric features:', numeric_features)
print('Found categorical features:', categorical_features)

Found numeric features: ['chroma_stft_var', 'chroma_stft_mean', 'chroma_cqt_var', 'chroma_cqt_mean', 'rmse_var', 'rmse_mean', 'spectral_centroid_var', 'spectral_centroid_mean', 'spectral_bandwidth_var', 'spectral_bandwidth_mean', 'rolloff_var', 'rolloff_mean', 'zero_crossing_rate_var', 'zero_crossing_rate_mean', 'mfcc1_var', 'mfcc1_mean', 'mfcc2_var', 'mfcc2_mean', 'mfcc3_var', 'mfcc3_mean', 'mfcc4_var', 'mfcc4_mean', 'mfcc5_var', 'mfcc5_mean', 'mfcc6_var', 'mfcc6_mean', 'mfcc7_var', 'mfcc7_mean', 'mfcc8_var', 'mfcc8_mean', 'mfcc9_var', 'mfcc9_mean', 'mfcc10_var', 'mfcc10_mean', 'mfcc11_var', 'mfcc11_mean', 'mfcc12_var', 'mfcc12_mean', 'mfcc13_var', 'mfcc13_mean', 'mfcc14_var', 'mfcc14_mean', 'mfcc15_var', 'mfcc15_mean', 'mfcc16_var', 'mfcc16_mean', 'mfcc17_var', 'mfcc17_mean', 'mfcc18_var', 'mfcc18_mean', 'mfcc19_var', 'mfcc19_mean', 'mfcc20_var', 'mfcc20_mean', 'mfcc21_var', 'mfcc21_mean', 'mfcc22_var', 'mfcc22_mean', 'mfcc23_var', 'mfcc23_mean', 'mfcc24_var', 'mfcc24_mean', 'mfcc25_

### (not standard scaler for RandomForestClassifier and XGBoostClassifier) ###

## Test different ML models ##

In [39]:
scores_df = pd.DataFrame(columns = ['model', 'accuracy', 'f1_score', 'set'])

### Train RandomForestClassifiers ###

In [36]:
# Perform grid search
print("Grid search...")
classifier = RandomForestClassifier(criterion = 'entropy')

# Grid of values to be tested
params = {
    "max_depth": [2, 4, 6, 8, 10],
    "min_samples_leaf": [1, 2, 5],
    "min_samples_split": [2, 4, 8, 10],
    "n_estimators": [10, 20, 40, 60, 80, 100],
}
gridsearch = GridSearchCV(
    classifier, param_grid=params, cv=3
)  # cv : the number of folds to be used for CV
gridsearch.fit(X_train, y_train_encoded)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)

Grid search...
...Done.
Best hyperparameters :  {'max_depth': 8, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 40}
Best validation accuracy :  0.591195917107859
Accuracy on training set :  0.0
Accuracy on test set :  0.0


#### Get predicictions on Train / val ####

In [37]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy on training set : ", gridsearch.score(X_train, y_train_encoded))
print("Accuracy on test set : ", gridsearch.score(X_val, y_val_encoded))

best_rf = gridsearch.best_estimator_

y_train_pred= best_rf.predict(X_train)
y_train_pred_classes = encoder.inverse_transform(y_train_pred)

y_val_pred= best_rf.predict(X_val)
y_val_pred_classes = encoder.inverse_transform(y_val_pred)
y_val_classes = encoder.inverse_transform(y_val_encoded)
print("\nClassification report:")
print(classification_report(y_val_classes, y_val_pred_classes))

#pd.Series(y_val_pred).value_counts()
pd.Series(y_val_pred_classes).value_counts()

Accuracy on training set :  0.8594224924012158
Accuracy on test set :  0.6121212121212121

Classification report:
                   precision    recall  f1-score   support

   diagnosis_adrd       0.63      0.28      0.39       104
diagnosis_control       0.61      0.91      0.73       183
    diagnosis_mci       0.50      0.16      0.25        43

         accuracy                           0.61       330
        macro avg       0.58      0.45      0.46       330
     weighted avg       0.60      0.61      0.56       330



,count
diagnosis_control,270
diagnosis_adrd,46
diagnosis_mci,14


In [38]:
# Print scores
print("accuracy on training set : ", accuracy_score(y_train_encoded, y_train_pred))
print("accuracy on test set : ", accuracy_score(y_val_encoded, y_val_pred))
print()

print("f1-score on training set : ", f1_score(y_train_encoded, y_train_pred, average='weighted'))
print("f1-score on test set : ", f1_score(y_val_encoded, y_val_pred,average='weighted'))
print()

accuracy on training set :  0.8594224924012158
accuracy on test set :  0.6121212121212121

f1-score on training set :  0.8536542266698914
f1-score on test set :  0.5602846731518742



In [40]:


#Register model metrics
new_rows = [
    {'model': 'random_forest', 'accuracy': gridsearch.score(X_train, y_train_encoded)
     ,'f1_score':f1_score(y_train_encoded, y_train_pred, average='weighted'), 'set': 'train'},
    {'model': 'random_forest', 'accuracy': gridsearch.score(X_val, y_val_encoded)
     ,'f1_score':f1_score(y_val_encoded, y_val_pred, average='weighted'), 'set': 'test'}
]

scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df

/tmp/ipython-input-1288206512.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)


,model,accuracy,f1_score,set
0,random_forest,0.859422,0.853654,train
1,random_forest,0.612121,0.560285,test


### Train XGBoostClassifier ###

In [41]:
# Perform grid search
print("Grid search...")
xgboost = XGBClassifier(objective='multi:softprob', num_class=len(CLASSES) )

# Grid of values to be tested
params = {
    'max_depth': [4, 6, 8, 10],
    'min_child_weight': [1, 2, 4, 6, 8],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}
print(params)
gridsearch_xgboost = GridSearchCV(xgboost, param_grid = params, cv = 3, verbose = 1) # cv : the number of folds to be used for CV
gridsearch_xgboost.fit(X_train, y_train_encoded)
print("...Done.")
print("Best hyperparameters : ", gridsearch_xgboost.best_params_)
print("Best validation accuracy : ", gridsearch_xgboost.best_score_)
print()
print("Accuracy on training set : ", gridsearch_xgboost.score(X_train, y_train_encoded))
print("Accuracy on test set : ", gridsearch_xgboost.score(X_val, y_val_encoded))

best_xgb = gridsearch_xgboost.best_estimator_

y_train_pred= best_xgb.predict(X_train)
y_train_pred_classes = encoder.inverse_transform(y_train_pred)

y_val_pred= best_xgb.predict(X_val)
y_val_pred_classes = encoder.inverse_transform(y_val_pred)
y_val_classes = encoder.inverse_transform(y_val_encoded)

print("\nClassification report:")
print(classification_report(y_val_classes, y_val_pred_classes))

new_rows = [
    {'model': 'xgboost', 'accuracy': gridsearch_xgboost.score(X_train, y_train_encoded)
       ,'f1_score':f1_score(y_train_encoded, y_train_pred, average='weighted'), 'set': 'train'},
    {'model': 'xgboost', 'accuracy': gridsearch_xgboost.score(X_val, y_val_encoded)
      ,'f1_score':f1_score(y_val_encoded, y_val_pred, average='weighted'), 'set': 'test'}
]

scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df

Grid search...
{'max_depth': [4, 6, 8, 10], 'min_child_weight': [1, 2, 4, 6, 8], 'n_estimators': [2, 4, 6, 8, 10, 12]}
Fitting 3 folds for each of 120 candidates, totalling 360 fits
...Done.
Best hyperparameters :  {'max_depth': 4, 'min_child_weight': 4, 'n_estimators': 6}
Best validation accuracy :  0.5759804176504647

Accuracy on training set :  0.7082066869300911
Accuracy on test set :  0.5969696969696969

Classification report:
                   precision    recall  f1-score   support

   diagnosis_adrd       0.57      0.23      0.33       104
diagnosis_control       0.61      0.91      0.73       183
    diagnosis_mci       0.40      0.14      0.21        43

         accuracy                           0.60       330
        macro avg       0.53      0.43      0.42       330
     weighted avg       0.57      0.60      0.54       330



,model,accuracy,f1_score,set
0,random_forest,0.859422,0.853654,train
1,random_forest,0.612121,0.560285,test
2,xgboost,0.708207,0.672051,train
3,xgboost,0.596970,0.536751,test


### (Optional) mp3 to wav conversion X ###

In [ ]:
! pip install pydub

In [ ]:
from pydub import AudioSegment

input_folder_test_path = TEST_AUDIOS_DIR

output_folder_test_path = TEST_AUDIOS_WAV_DIR

def convert_mp3_to_wav(input_file_path, output_file_path):

    audio_mp3 = AudioSegment.from_file(input_file_path, format="mp3")

    audio_mp3.export(output_file_path, format="wav")
    print(f"Converted : {input_file_path} -> {output_file_path}")

def convert_mp3_to_wav_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for file_name in os.listdir(input_folder):
        if file_name.endswith(".mp3"):
            input_file_path = os.path.join(input_folder, file_name)
            output_file_name = file_name.replace(".mp3", ".wav")
            output_file_path = os.path.join(output_folder, output_file_name)

            if os.path.exists(output_file_path):
                print(f"Already converted: {output_file_path}, conversion ignorée.")
            else:
                convert_mp3_to_wav(input_file_path, output_file_path)

'''process_folder(input_folder_test_path, output_folder_test_path)
process_folder(input_folder_train_path, output_folder_train_path)
'''
convert_mp3_to_wav_folder(input_folder_test_path, output_folder_test_path)

audio_files = glob(TEST_AUDIOS_WAV_DIR +'*.wav')
print(len(audio_files))
print(audio_files[0])
print(type(audio_files[0]))

ModuleNotFoundError: No module named 'pydub'

### X Test pre Processing ###

In [ ]:
def process_X_test_features(audio_file_dir:str, sound_processing_config: Sound_processing_config) -> pd.DataFrame:
  """
  """
  dict_list = list()
  i = 0
  files = glob(audio_file_dir +'*.wav')
  for file_path in files:
    i+=1
    dict_list.append(extract_audio_features(file_path, sound_processing_config))
    print('% done : ', i/len(files)*100)
  return pd.DataFrame(dict_list)

X_test = process_X_test_features(TEST_AUDIOS_WAV_DIR, sound_processing_config)
X_test.head(5)
X_test_uid = X_test

TypeError: 'module' object is not callable

#### X Test transform ####

In [ ]:
X_test = X_test.drop(columns=['uid'], axis=1)
X_test = preprocessor.transform(X_test)

#### Predictions on Test ####

In [ ]:
y_test_pred = gridsearch.predict(X_test)
y_test_pred_proba = gridsearch.predict_proba(X_test)
y_test_pred_classes = encoder.inverse_transform(y_test_pred)
pd.Series(y_test_pred_classes).value_counts()

array([1, 1, 1, 1, 1, 1, 0, 1, 2, 1])

#### Process Y test predictions with required structure / format for Challenge ####

In [ ]:
dtf = pd.DataFrame(columns=['class'],data=y_test_pred_classes)
dtf = pd.merge(X_test_uid, dtf,left_index=True, right_index=True)
dtf = dtf[['uid','class']]
dtf['diagnosis_mci'] = np.where(dtf['class'] == 'diagnosis_mci',1,0)
dtf['diagnosis_adrd'] = np.where(dtf['class'] == 'diagnosis_adrd',1,0)
dtf['diagnosis_control'] = np.where(dtf['class'] == 'diagnosis_control',1,0)
dtf = dtf.drop(columns=['class'], axis=1)
dtf.head(10)

<class 'pandas.core.frame.DataFrame'>


,uid,diagnosis_mci,diagnosis_adrd,diagnosis_control
0,fixv,0,0,1
1,gphn,0,0,1
2,ibzi,0,0,1
3,amhc,0,0,1
4,zllm,0,0,1
5,goof,0,0,1
6,qpyw,0,1,0
7,beqf,0,0,1
8,ydew,1,0,0
9,uvnq,0,0,1


#### Saving Y pred dataframe to csv file ####

In [ ]:
dtf = dtf.loc[:,['uid','diagnosis_control','diagnosis_mci','diagnosis_adrd']]
SUBMISSION_TEST_FILE_PATH = os.path.join(PROJECT_DIR,'Submissions','submission_test_20241218.csv')
dtf.sort_values(by='uid', ascending=True, inplace=True)
dtf = dtf.astype({'diagnosis_control':'float64','diagnosis_mci':'float64','diagnosis_adrd':'float64'})
dtf.to_csv(SUBMISSION_TEST_FILE_PATH, index=False)

dtf.head(10)

,uid,diagnosis_control,diagnosis_mci,diagnosis_adrd
411,aazd,1.0,0.0,0.0
363,acdn,1.0,0.0,0.0
240,adoi,1.0,0.0,0.0
248,agiy,1.0,0.0,0.0
237,agni,1.0,0.0,0.0
383,agon,1.0,0.0,0.0
16,aikh,0.0,1.0,0.0
144,ajef,0.0,0.0,1.0
23,ajxm,1.0,0.0,0.0
153,amdz,1.0,0.0,0.0
